# Construction de l'index FAISS

**Objectif :** découper les descriptions en chunks, les embedder avec `mistral-embed` et construire un index FAISS via LangChain.

Étapes :
1. Chargement des données traitées
2. Chunking — découpage de `longdescription_fr` en morceaux sémantiques
3. Création des Documents LangChain (chunk + métadonnées)
4. Construction de l'index FAISS par batch
5. Sauvegarde de l'index

In [ ]:
import os
import time
import pandas as pd
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

df = pd.read_csv("../data/processed/events_clean.csv")
print(f"{len(df)} événements chargés")

5030 événements chargés


## 2. Stratégie d'indexation : 1 événement = 1 Document

### Pourquoi pas de chunking ici ?

Le chunking est recommandé pour des **documents longs** (articles, rapports, pages web) où un seul vecteur ne peut pas capturer toute la sémantique.

Pour notre dataset, ce n'est pas le cas :
- Les descriptions d'événements sont courtes (200–800 caractères en moyenne)
- Chaque événement est déjà une **unité sémantique atomique** — le découper fragmente du sens sans bénéfice
- Avec k=5 événements récupérés, le contexte total (~5 000 caractères) est très loin des limites de Mistral (32k tokens)

On indexe donc **1 Document LangChain par événement**, avec l'ensemble des informations dans `page_content` et les métadonnées dans `metadata`.

In [ ]:
df = df.dropna(subset=["longdescription_fr"])
# un même événement peut apparaître dans plusieurs agendas OpenAgenda → déduplique sur uid
df = df.drop_duplicates(subset=["uid"])
print(f"{len(df)} événements après déduplication")

documents = []
for _, row in df.iterrows():
    metadata = {
        "uid": row["uid"],
        "title": row["title_fr"],
        "conditions": row["conditions_fr"],
        "date_start": row["firstdate_begin"],
        "date_end": row["lastdate_end"],
        "location": f"{row['location_name']}, {row['location_address']}",
        "url": row["canonicalurl"],
    }
    # page_content = corpus sémantique (titre + description uniquement, sans métadonnées)
    documents.append(Document(page_content=row["corpus"], metadata=metadata))

print(f"{len(documents)} documents créés (1 par événement)")

## 3. Initialisation du modèle d'embedding

On initialise `mistral-embed`. La dimension des vecteurs produits est 1024.

In [7]:
embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

## 4. Construction de l'index FAISS par batch

On utilise `FAISS.from_documents()` de LangChain qui gère automatiquement le stockage des métadonnées avec chaque vecteur.  
Pour respecter le rate limiting Mistral, on construit l'index par batch de 50 documents avec une pause entre chaque.

In [8]:
batch_size = 50
faiss_store = None

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    if faiss_store is None:
        faiss_store = FAISS.from_documents(batch, embed_model)
    else:
        faiss_store.add_documents(batch)
    print(f"{min(i + batch_size, len(documents))}/{len(documents)}", end="\r")
    time.sleep(1)

print(f"\nIndex construit : {faiss_store.index.ntotal} vecteurs")

os.makedirs("../data/faiss_index", exist_ok=True)
faiss_store.save_local("../data/faiss_index")
print("Index sauvegardé dans data/faiss_index/")

5028/5028
Index construit : 5028 vecteurs
Index sauvegardé dans data/faiss_index/
